In [0]:
%run ./utils

In [0]:
from datetime import datetime, timedelta
import pyspark.sql.functions as F

def get_checksum_df(check_topics: str, start_ms: int, end_ms: int, kafka_brokers: str):
    """
    check_topics: 例如 "Consumer_AU,Consumer_HK,Consumer_AU_TS"
    start_ms/end_ms: 毫秒时间戳（Kafka读取范围）
    """

    if not check_topics or not check_topics.strip():
        raise ValueError("check_topics is empty")
    if start_ms >= end_ms:
        raise ValueError("start_ms must be less than end_ms")

    selected_keys = [x.strip() for x in check_topics.split(",") if x.strip()]
    unknown_keys = [k for k in selected_keys if k not in TOPIC_GROUPS]
    if unknown_keys:
        raise ValueError(f"Unknown check_topics: {unknown_keys}")

    # 组装 pair
    pair_rows = []
    for k in selected_keys:
        raw_topic, validated_topic = TOPIC_GROUPS[k]
        pair_rows.append((k, raw_topic, validated_topic))

    # subscribe_topics 仅由 check_topics 决定
    subscribe_topics = sorted({t for _, r, v in pair_rows for t in (r, v)})
    subscribe_topics_str = ",".join(subscribe_topics)

    # Kafka按时间范围读取（元数据时间）
    kafka_df = (
        spark.read
        .format("kafka")
        .option("kafka.bootstrap.servers", kafka_brokers)
        .option("subscribe", subscribe_topics_str)
        .option("startingTimestamp", str(int(start_ms)))
        .option("endingTimestamp", str(int(end_ms)))
        .option("startingOffsetsByTimestampStrategy", "latest")
        .load()
    )

    # 再次过滤，确保窗口精确
    start_ts = F.expr(f"timestamp_millis({int(start_ms)})")
    end_ts = F.expr(f"timestamp_millis({int(end_ms)})")
    kafka_df = kafka_df.filter((F.col("timestamp") >= start_ts) & (F.col("timestamp") < end_ts))

    # topic计数
    topic_cnt_df = kafka_df.groupBy("topic").agg(F.count(F.lit(1)).cast("long").alias("cnt"))

    # pair映射DF
    schema = T.StructType([
        T.StructField("check_topic", T.StringType(), False),
        T.StructField("raw_topic", T.StringType(), False),
        T.StructField("validated_topic", T.StringType(), False),
    ])
    map_df = spark.createDataFrame(pair_rows, schema=schema)

    raw_df = topic_cnt_df.select(F.col("topic").alias("raw_topic"), F.col("cnt").alias("raw_count"))
    val_df = topic_cnt_df.select(F.col("topic").alias("validated_topic"), F.col("cnt").alias("validated_count"))

    check_df = (
        map_df
        .join(raw_df, on="raw_topic", how="left")
        .join(val_df, on="validated_topic", how="left")
        .fillna(0, subset=["raw_count", "validated_count"])
        .withColumn("ratio_value", F.when(F.col("raw_count") > 0,  F.abs(F.col("raw_count") - F.col("validated_count")) / F.col("raw_count")).otherwise(F.lit(0.0)))
        .withColumn(
            "is_need_monitor",
            F.when((F.col("raw_count") == 0) & (F.col("validated_count") > 0), F.lit(True))   # 告警
             .when((F.col("raw_count") > 0) & (F.col("validated_count") == 0), F.lit(True))    # 告警
             .when((F.col("raw_count") == 0) & (F.col("validated_count") == 0), F.lit(False))  # 双零不告警
             .otherwise(F.col("ratio_value") >= F.lit(0.8))                                      # 固定阈值
        )
        .withColumn(
            "rule_hit",
            F.when((F.col("raw_count") == 0) & (F.col("validated_count") > 0), F.lit("raw0_val_gt0"))
             .when((F.col("raw_count") > 0) & (F.col("validated_count") == 0), F.lit("raw_gt0_val0"))
             .when((F.col("raw_count") == 0) & (F.col("validated_count") == 0), F.lit("both_zero"))
             .otherwise(F.lit("ratio_check"))
        )
        .select(
            "check_topic", "raw_topic", "validated_topic",
            "raw_count", "validated_count", "ratio_value",
            "is_need_monitor", "rule_hit"
        )
        .orderBy(F.col("ratio_value").desc(), F.col("check_topic").asc())
    )

    return check_df


In [0]:
def monitor_main(monitor_id, check_topics: str, start_ms: int, end_ms: int, kafka_brokers: str, end_time: datetime, max_rows=MAX_ROWS, max_cols=MAX_COLS, to_addrs=None):
    # 1. check sum
    check_df = get_checksum_df(check_topics, start_ms, end_ms, kafka_brokers)
    check_df.cache()
    display(check_df)

    if check_df.filter(F.col("is_need_monitor") == True).count() > 0:
        print(f"This inspection found invalid data: {monitor_id}")
        display(check_df)

        # 2. build email body
        html_body = build_html_table_from_spark_df(check_df, max_rows=max_rows, max_cols=max_cols)

        recipients = to_addrs or TO_ADDRS
        if not recipients:
            raise ValueError("to_addrs is empty; no recipients configured for the topic exception report email.")

        send_email(
            subject=SUBJECT.format(yyyymmdd=end_time.strftime("%Y%m%d")),
            html_body=html_body,
            to_addrs=recipients,
            cc_addrs=CC_ADDRS,
            bcc_addrs=BCC_ADDRS,
            custom_text = f"This inspection found topic exception.  <br>Check time period(UTC): {start_time} -> {end_time}.  <br>monitor_id: {monitor_id}"
        )

    else:
        print(f"There is no invalid data in this check: {monitor_id}")

    check_df.unpersist()

In [0]:
TOPIC_GROUPS = {
    "ConsumerTopic_VL": ("ConsumerTopic", "ConsumerTopic_VL"),
    "Consumer_HK_VL": ("Consumer_HK", "Consumer_HK_VL"),
    "Consumer_JP_VL": ("Consumer_JP", "Consumer_JP_VL"),
    "Consumer_KR_VL": ("Consumer_KR", "Consumer_KR_VL"),
    "Consumer_MY_VL": ("Consumer_MY", "Consumer_MY_VL"),
    "Consumer_NZ_VL": ("Consumer_NZ", "Consumer_NZ_VL"),
    "Consumer_PH_VL": ("Consumer_PH", "Consumer_PH_VL"),
    "Consumer_SG_VL": ("Consumer_SG", "Consumer_SG_VL"),
    "Consumer_TH_VL": ("Consumer_TH", "Consumer_TH_VL"),
    "Consumer_TW_VL": ("Consumer_TW", "Consumer_TW_VL"),
    "Consumer_VN_VL": ("Consumer_VN", "Consumer_VN_VL"),
    "Consumer_ID_VL": ("Consumer_ID", "Consumer_ID_VL"),
    "Consumer_JP_Rakuten_VL": ("Consumer_JP_Rakuten", "Consumer_JP_Rakuten_VL"),
    "Consumer_TW_Linegift_VL": ("Consumer_TW_Linegift", "Consumer_TW_Linegift_VL"),

    "Consumer_AU_TS_Validated": ("Consumer_AU_TS", "Consumer_AU_TS_Validated"),
    "Consumer_HK_TS_Validated": ("Consumer_HK_TS", "Consumer_HK_TS_Validated"),
    "Consumer_JP_TS_Validated": ("Consumer_JP_TS", "Consumer_JP_TS_Validated"),
    "Consumer_KR_TS_Validated": ("Consumer_KR_TS", "Consumer_KR_TS_Validated"),
    "Consumer_MY_TS_Validated": ("Consumer_MY_TS", "Consumer_MY_TS_Validated"),
    "Consumer_NZ_TS_Validated": ("Consumer_NZ_TS", "Consumer_NZ_TS_Validated"),
    "Consumer_PH_TS_Validated": ("Consumer_PH_TS", "Consumer_PH_TS_Validated"),
    "Consumer_SG_TS_Validated": ("Consumer_SG_TS", "Consumer_SG_TS_Validated"),
    "Consumer_TH_TS_Validated": ("Consumer_TH_TS", "Consumer_TH_TS_Validated"),
    "Consumer_TW_TS_Validated": ("Consumer_TW_TS", "Consumer_TW_TS_Validated"),
    "Consumer_VN_TS_Validated": ("Consumer_VN_TS", "Consumer_VN_TS_Validated"),
    "Consumer_ID_TS_Validated": ("Consumer_ID_TS", "Consumer_ID_TS_Validated"),
    "Consumer_JP_Rakuten_TS_Validated": ("Consumer_JP_Rakuten_TS", "Consumer_JP_Rakuten_TS_Validated"),
    "Consumer_TW_Linegift_TS_Validated": ("Consumer_TW_Linegift_TS", "Consumer_TW_Linegift_TS_Validated"),
}



TO_ADDRS: List[str] = []
CC_ADDRS: List[str] = []
BCC_ADDRS: List[str] = []

SUBJECT = "[Critical] [MDM] Topic Records Discrepancy >80% {yyyymmdd}"

In [0]:
monitor_id = dbutils.widgets.get("monitor_id")
hour_time_period = int(dbutils.widgets.get("hour_time_period"))
check_topics = dbutils.widgets.get("check_topics")
kafka_brokers = dbutils.widgets.get("kafka_brokers")
topic_groups_json = dbutils.widgets.get("topic_groups_json")

if topic_groups_json and topic_groups_json.strip():
    import json
    # widget 传 JSON string，格式: {"key": ["raw_topic", "validated_topic"], ...}
    TOPIC_GROUPS = json.loads(topic_groups_json)

try:
    trigger_timestamp_ms = int(dbutils.widgets.get("trigger_timestamp_ms")) / 1000
except:
    trigger_timestamp_ms = int(datetime.now().timestamp())

# Maximum rows/columns to show in the email HTML tables.
try:
    max_rows = int(dbutils.widgets.get("max_rows"))
except:
    max_rows = MAX_ROWS

try:
    max_cols = int(dbutils.widgets.get("max_cols"))
except:
    max_cols = MAX_COLS

# Comma-separated list of recipient email addresses.
to_addrs_str = dbutils.widgets.get("to_addrs")
to_addrs = [x.strip() for x in to_addrs_str.split(",") if x.strip()] if to_addrs_str else TO_ADDRS

end_time = datetime.fromtimestamp(trigger_timestamp_ms)
start_time = end_time - timedelta(hours= hour_time_period)

start_ms = int(start_time.timestamp() * 1000)
end_ms = int(end_time.timestamp() * 1000)

print(f"monitor_id: {monitor_id}")
print(f"check_topics: {check_topics}")
print(f"kafka_brokers: {kafka_brokers}")
print(f"hour_time_period: {hour_time_period}")
print(f"max_rows: {max_rows}, max_cols: {max_cols}")
print(f"to_addrs: {to_addrs}")
print(f"start_time: {start_time}, end_time: {end_time}")
print(f"start_ms: {start_ms}, end_ms: {end_ms}")

monitor_main(monitor_id, check_topics, start_ms, end_ms, kafka_brokers, end_time, max_rows, max_cols, to_addrs)